# Transfer Learning

Is the process os re-using the acquired weights of a model alreadey trained (usually in a similar task) in a custom model, allowing it to reach better levels with less training time
- The pre-trained model is usually called Foundation Model

---
Get model weights at https://huggingface.co/models
torch libraries:
- torchvision.models
- torchtext.models
- torchaudio.models
- torchrec.models

In [ ]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
device = torch.device(device)
BATCH_SIZE = 32

# !nvidia-smi

## Getting the Data

In [14]:
from src.helper_functions import download_data

img_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",destination="pizza_steak_sushi")
train_dir = img_path / "train"
test_dir = img_path / "test"

## Turning raw Data into Datasets and Dataloaders

from torchvision 0.13 onwards, there are 2 ways about it
- Manually:
  - The programmer defines the transformations
- Automatically:
  - The model iteratively decides which transformations to use

For transfer learning, it is very important that the transformations in the custom model are the same of the Foundation Model

In [22]:
from torchvision.models import EfficientNet_B0_Weights as ww

# Get a set of pretrained model weights
weights = ww.DEFAULT # .DEFAULT = best available weights from pretraining on ImageNet

### Manual Transformation

Using torchvision.models, it's noted that all pre-trained models expect input images nomalized the same way
- mini-batches of 3-channel RGB
- shape of (3 x H x W)
- H and W must be at least 224
- The images must be loaded in [0, 1] range
  - Then normalized  with:
  - `mean = [0.485, 0.456, 0.406]`
  - `std = [0.229, 0.224, 0.225]`

In [23]:
from torchvision import transforms as T

manual_transform = T.Compose(
    [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ]
)
manual_transform

### Automatic Transformations

In [ ]:
auto_transforms = weights.transforms()

auto_transforms

### Creating the Dataloaders

Using the automatic transform, as the manual one is different

In [ ]:
from src.data_setup import create_img_dataloaders

train_dataloader, test_dataloader, class_names = create_img_dataloaders(
    train_dir=str(train_dir),
    test_dir=str(test_dir),
    transform=auto_transforms, # manual_transform,
    batch_size=BATCH_SIZE,
)

# Choosing a Foundation Model

Main points to consider
1. Speed
   1. How fast can the model replication be
   2. Does the desired use case requires speed?
2. Size
   1. How much memory will the model replication need
   2. Does the desired use case requires speelow memory footprint?
   3. usually tied with the speed, but not always
3. Performance
   1. How well it performs in the chosen problem domain
   2. How much performance can be obtained without losing applicability on the 2 above factors?

In [24]:
# Pre 0.13 torchvision version:
from torchvision.models import efficientnet_b0 as b0
from torchvision.models import EfficientNet_B0_Weights as w0

# model = b0(pretrained=True).to(device)

# After 0.13 torchvision version:
weights = w0.DEFAULT
model = b0(weights=weights).to(device)

model

## Adapting the Foundation Model

The pre-trained model won't always have the same output shape of the custom model
* Therefore, there are different approaches, types of Transfer Learning, to appy:

1. Original Model (simply using the Foundation Model without re-training)
2. Feature Extraction
3. Fine Tunning

### Feature Extraction

"extract" some of the Foundation Model' layers
* The input and hidden layers become "frozen" during training
* only the output layer is fitted to the desired problem

### Fine Tunning

Usually complements the Feature Extraction
* Better suited when the target problem has large ammounts of data
* Some, many or all the layers of the Foundation Model are updated during the training